# Lesson 12: Reasoning models and test-time compute

*Companion notebook for the lesson page. Worked solutions are in `solutions/12-reasoning-models-solutions.ipynb`; try each exercise first.*

Use models that think before they answer: what test-time compute buys, how reasoning models are trained, how to call them with thinking and effort controls, how to build self-consistency, best-of-N and verifier loops yourself, and how to prompt, evaluate and route reasoning systems without overpaying.

**Runs on:** Any computer (CPU) for the simulations. Cells that call a model need an API key and are marked.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy matplotlib anthropic pydantic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()   # reads ANTHROPIC_API_KEY from a .env file next to this notebook, if there is one
# On Google Colab, add the key under the key icon (Secrets), then uncomment the next two lines:
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()
MODEL = "claude-sonnet-5-5"          # a capable model with thinking: check your provider's current model list
MODEL_FAST = "claude-haiku-5-5"      # a fast, cheap model for the routing examples
print("API key found" if os.environ.get("ANTHROPIC_API_KEY") else
      "No API key set: cells marked 'Needs an API key' will fail until you add one")

## Thinking is computation

Prompted chain-of-thought works because every generated token is another pass of computation that can use the tokens before it. Reasoning models are trained so that this scratch space is used well, and they decide how long to think within a budget you set.

## Test-time compute: the second scaling axis

Spend more per question (longer thinking, several attempts, a verifier) instead of, or as well as, a bigger model. Session 2 builds the techniques; this session looks at what the API gives you.

## Thinking in an API

`client`, `MODEL` and `MODEL_FAST` come from the setup cell. Read content blocks by type, never by position, and remember that thinking counts against `max_tokens`.

In [ ]:
# Needs an API key (see the setup cell)
question = ("A sensor logs one reading every 15 seconds and 5% of readings fail a "
            "quality check. How many good readings does it keep in 6 hours?")

resp = client.messages.create(
    model=MODEL,
    max_tokens=8000,                                         # thinking counts against this cap
    thinking={"type": "adaptive", "display": "summarized"},  # return a readable summary of the thinking
    output_config={"effort": "medium"},                      # low | medium | high | xhigh | max
    messages=[{"role": "user", "content": question}],
)
for block in resp.content:                                   # read blocks by type, never by position
    if block.type == "thinking":
        print("THINKING:", block.thinking[:400], "...\n")
    elif block.type == "text":
        print("ANSWER:", block.text)
print(resp.stop_reason, "| input", resp.usage.input_tokens,
      "| output incl. thinking", resp.usage.output_tokens)

> **Thinking controls change with every model generation.** At the time of writing, current Claude models take `thinking={"type": "adaptive"}` and `output_config={"effort": ...}` and reject the older `budget_tokens` setting and non-default temperatures. Keep the model name and its thinking settings together in one place and read the provider's current documentation when you change models.

### Watching it think

Stream, and show the thinking summary and the answer as they arrive instead of a blank screen.

In [ ]:
# Needs an API key (see the setup cell)
with client.messages.stream(
    model=MODEL, max_tokens=8000,
    thinking={"type": "adaptive", "display": "summarized"},
    output_config={"effort": "medium"},
    messages=[{"role": "user", "content": question}],
) as stream:
    for event in stream:
        if event.type == "content_block_start":
            print(f"\n[{event.content_block.type}]")
        elif event.type == "content_block_delta":
            if event.delta.type == "thinking_delta":
                print(event.delta.thinking, end="", flush=True)
            elif event.delta.type == "text_delta":
                print(event.delta.text, end="", flush=True)
    final = stream.get_final_message()
print("\n", final.usage.output_tokens, "output tokens")

### Effort, cost and latency

Thinking tokens are output tokens, and all of them are generated before the answer starts. The `ask()` helper below is reused in sessions 2 and 3. Set the prices to your provider's current list.

In [ ]:
# Needs an API key (see the setup cell)
import time

PRICE_IN, PRICE_OUT = 2.0, 10.0     # USD per million tokens at the time of writing: check the current price list

def ask(question, effort="medium", max_tokens=8000, model=None):
    """One call with adaptive thinking. Returns (visible text, usage, seconds)."""
    t0 = time.time()
    resp = client.messages.create(
        model=model or MODEL, max_tokens=max_tokens,
        thinking={"type": "adaptive"},
        output_config={"effort": effort},
        messages=[{"role": "user", "content": question}],
    )
    text = "".join(b.text for b in resp.content if b.type == "text")
    if resp.stop_reason == "max_tokens":
        text += "\n[cut off: the budget went on thinking; raise max_tokens or lower effort]"
    return text, resp.usage, time.time() - t0

def cost(usage):
    return usage.input_tokens / 1e6 * PRICE_IN + usage.output_tokens / 1e6 * PRICE_OUT

for effort in ["low", "medium", "high"]:
    text, usage, secs = ask(question, effort=effort)
    last_line = text.strip().splitlines()[-1][:60] if text.strip() else "(no text)"
    print(f"{effort:>6}: {usage.output_tokens:>5} output tokens, {secs:4.1f}s, "
          f"${cost(usage):.4f}  ->  {last_line}")

Watch the output tokens and the seconds climb with effort while the final line stays the same: on an easy question you pay for thinking the question did not need.

## When reasoning helps, and when it does not

Multi-step arithmetic, code, planning under constraints and evidence-gathering analysis gain the most. Lookups, classification, extraction and latency-critical paths gain little or nothing: route those to a fast model at low effort.

### Hands-on: when does thinking pay?

1. Write six questions with checkable answers: three one-step lookups and three multi-step word problems. Ask for the answer in a fixed format so a small `grade()` function can mark it.
2. Run each question at effort `low` and `high` with `ask()`, recording whether it was right, output tokens and seconds.
3. For each (class, effort) pair compute cost per correct answer.
4. You are done when you can state, with numbers, which class should run at which effort.

#### Solution: an effort sweep by question class

In [ ]:
# Needs an API key (see the setup cell)
import re

easy = [("What is the capital of Peru? Reply with the city name only.", "lima"),
        ("How many minutes are there in a day? Reply with the number only.", "1440"),
        ("What is the chemical symbol for sodium? Reply with the symbol only.", "na")]
hard = [("A pump moves 18 litres a minute for 2.5 hours, then 12 litres a minute for 45 minutes. "
         "How many litres in total? End with 'Answer: <number>'.", "3240"),
        ("A train leaves at 09:40 and arrives at 14:05 the same day. It stops for a total of 35 minutes. "
         "For how many minutes is it moving? End with 'Answer: <number>'.", "230"),
        ("A 1.2 kW heater runs 4 hours a night for 30 nights. Electricity costs 0.25 per kWh. "
         "What is the total cost? End with 'Answer: <number>'.", "36")]

def grade(text, gold):
    tokens = re.findall(r"[a-z]+|-?\d[\d,]*\.?\d*", text.lower())
    tokens = [t.replace(",", "").removesuffix(".0") for t in tokens]
    return gold in tokens

for label, qs in [("easy", easy), ("hard", hard)]:
    for effort in ["low", "high"]:
        right, tokens, dollars, secs = 0, 0, 0.0, 0.0
        for q, gold in qs:
            text, usage, s = ask(q, effort=effort)
            right += grade(text, gold); tokens += usage.output_tokens
            dollars += cost(usage); secs += s
        per_correct = dollars / right if right else float("inf")
        print(f"{label} / {effort:>4}: {right}/{len(qs)} correct, {tokens:>5} output tokens, "
              f"{secs:4.1f}s, ${dollars:.4f} total, ${per_correct:.4f} per correct answer")

What to observe: on the lookups both effort levels are usually 3/3 and the low-effort rows use far fewer output tokens and seconds, so cost per correct answer is several times lower. On the word problems the result can go either way with a current model; the token and time columns always grow with effort. If every row is 3/3, make the hard questions harder until a difference appears.

---

## Session 2: Test-time compute you can build yourself

Treat the model as a noisy solver: sample it several times and choose well among the samples. Simulate first, then pay for real calls.

### Self-consistency: sample many, vote

Let `p` be the probability a sample is correct and `q` the probability of the most common wrong answer. Majority voting drives accuracy towards 1 if `p > q` and towards 0 if `p < q`.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def vote_accuracy(p, wrong, n, trials=20000):
    """Majority vote over n independent samples. Index 0 is the correct answer, drawn
    with probability p; the wrong answers 1..k are drawn with the probabilities in `wrong`."""
    probs = np.array([p, *wrong])
    answers = rng.choice(len(probs), size=(trials, n), p=probs)
    counts = (answers[:, :, None] == np.arange(len(probs))).sum(axis=1)   # votes per answer
    counts = counts + rng.random(counts.shape) * 0.5                      # break ties at random
    return np.mean(counts.argmax(axis=1) == 0)

spread = [0.1, 0.1, 0.1, 0.1]      # p = 0.6: wrong answers spread thinly over four distractors
close = [0.3, 0.2, 0.15]          # p = 0.35: the most common wrong answer is nearly as likely
lumped = [0.5, 0.1, 0.05]         # p = 0.35: one wrong answer is MORE likely than the right one
for n in [1, 3, 5, 9, 15, 25, 51]:
    print(f"N={n:>2}   spread {vote_accuracy(0.6, spread, n):.3f}   "
          f"close {vote_accuracy(0.35, close, n):.3f}   lumped {vote_accuracy(0.35, lumped, n):.3f}")

```text
N= 1   spread 0.597   close 0.354   lumped 0.346
N= 3   spread 0.718   close 0.379   lumped 0.337
N= 5   spread 0.838   close 0.413   lumped 0.338
N= 9   spread 0.939   close 0.463   lumped 0.309
N=15   spread 0.986   close 0.510   lumped 0.263
N=25   spread 0.999   close 0.566   lumped 0.206
N=51   spread 1.000   close 0.652   lumped 0.119
```

Three regimes: spread errors (voting is almost free accuracy), a close race (slow gains) and a lumped, systematic error (voting makes it worse).

In [ ]:
import matplotlib.pyplot as plt

ns = [1, 3, 5, 9, 15, 25, 51]
fig, ax = plt.subplots(figsize=(6, 3.5))
for label, p, wrong in [("spread (p=0.6)", 0.6, spread), ("close (p=0.35)", 0.35, close), ("lumped (p=0.35)", 0.35, lumped)]:
    ax.plot(ns, [vote_accuracy(p, wrong, n, trials=5000) for n in ns], marker="o", label=label)
ax.set_xscale("log"); ax.set_xlabel("N samples"); ax.set_ylabel("majority-vote accuracy")
ax.set_title("Self-consistency: three regimes"); ax.legend(); plt.tight_layout(); plt.show()

### Best-of-N with a verifier

A verifier scores every candidate and the top score wins. Its margin `d` decides how fast accuracy rises; its blind spots decide where more sampling starts to hurt.

In [ ]:
def best_of_n_accuracy(p, n, d, blind=0.0, trials=20000):
    """Sample n answers, let a verifier score each one and keep the top score.
    Correct answers score N(d, 1) and wrong ones N(0, 1), so d is how well the
    verifier separates them. A fraction `blind` of wrong answers fool it completely
    and score N(d + 3, 1)."""
    correct = rng.random((trials, n)) < p
    fooled = ~correct & (rng.random((trials, n)) < blind)
    scores = rng.normal(0, 1, size=(trials, n)) + d * correct + (d + 3) * fooled
    picked = scores.argmax(axis=1)
    return np.mean(correct[np.arange(trials), picked])

for n in [1, 2, 4, 8, 16, 32, 64]:
    print(f"N={n:>2}   weak verifier {best_of_n_accuracy(0.3, n, 1.0):.3f}   "
          f"strong {best_of_n_accuracy(0.3, n, 3.0):.3f}   "
          f"strong with blind spots {best_of_n_accuracy(0.3, n, 3.0, blind=0.03):.3f}")

```text
N= 1   weak verifier 0.304   strong 0.303   strong with blind spots 0.301
N= 2   weak verifier 0.411   strong 0.502   strong with blind spots 0.497
N= 4   weak verifier 0.524   strong 0.738   strong with blind spots 0.691
N= 8   weak verifier 0.615   strong 0.921   strong with blind spots 0.784
N=16   weak verifier 0.705   strong 0.988   strong with blind spots 0.720
N=32   weak verifier 0.764   strong 0.998   strong with blind spots 0.543
N=64   weak verifier 0.819   strong 1.000   strong with blind spots 0.325
```

In [ ]:
ns = [1, 2, 4, 8, 16, 32, 64]
fig, ax = plt.subplots(figsize=(6, 3.5))
for label, kw in [("weak verifier (d=1)", dict(d=1.0)), ("strong (d=3)", dict(d=3.0)),
                  ("strong with blind spots", dict(d=3.0, blind=0.03))]:
    ax.plot(ns, [best_of_n_accuracy(0.3, n, trials=5000, **kw) for n in ns], marker="o", label=label)
ax.set_xscale("log"); ax.set_xlabel("N samples"); ax.set_ylabel("best-of-N accuracy")
ax.set_title("Best-of-N against a verifier"); ax.legend(); plt.tight_layout(); plt.show()

The third column peaks around N = 8 and then falls: sampling harder against an imperfect judge finds the judge's mistakes. Measure your verifier's precision before you build best-of-N.

### Outcome versus process: verify the steps

With `s` steps each right with probability `q`, checking whole solutions succeeds with `1 - (1 - q^s)^N`; checking every step with `k` candidates per step succeeds with `(1 - (1 - q)^k)^s`.

In [ ]:
def outcome_search(q, steps, n):
    """Sample n complete solutions; a perfect outcome verifier keeps a correct one if any exists."""
    return 1 - (1 - q ** steps) ** n

def process_search(q, steps, k):
    """At every step sample k candidates; a perfect process verifier keeps a correct one if any exists."""
    return (1 - (1 - q) ** k) ** steps

for budget in [1, 2, 4, 8]:
    print(f"{budget} candidate(s) per step:  outcome search {outcome_search(0.8, 6, budget):.3f}   "
          f"process search {process_search(0.8, 6, budget):.3f}")

```text
1 candidate(s) per step:  outcome search 0.262   process search 0.262
2 candidate(s) per step:  outcome search 0.456   process search 0.783
4 candidate(s) per step:  outcome search 0.704   process search 0.990
8 candidate(s) per step:  outcome search 0.912   process search 1.000
```

Step-level feedback is dense feedback. The price is the process verifier: expensive labels, a noisy learned model, more search code.

### The same experiment with a real model

Ten hand-written arithmetic word problems with integer answers, and a parser for the final number. This cell runs offline.

In [ ]:
import re

PROBLEMS = [
    ("A sensor logs one reading every 15 seconds. How many readings does it log in 6 hours?", 1440),
    ("A battery stores 2400 Wh. An instrument draws 30 W continuously. For how many hours can it run before the battery is empty?", 80),
    ("A campaign budget is 5000. Each site visit costs 350 and each data upload costs 12. The team made 9 visits and 60 uploads. How much budget is left?", 1130),
    ("Three air samplers each run for 4 hours at 1.5 litres per minute. How many litres of air do they sample in total?", 1080),
    ("A dataset has 1250 rows. 8% of them fail a quality check and are removed. The remaining rows are split 80/20 into training and test sets. How many rows are in the test set?", 230),
    ("A spectrometer takes a 12-second exposure followed by a 3-second readout, then starts the next exposure. How many complete exposures fit in 20 minutes?", 80),
    ("A shipment of 7 crates weighs 2.1 tonnes in total. Each empty crate weighs 40 kg. What is the mass of the contents in kg?", 1820),
    ("Node A sends 48 packets a minute and node B sends 36. Each packet is 250 bytes. How many kilobytes (1 kB = 1000 bytes) do they send together in one hour?", 1260),
    ("A station has 4 instruments that each produce 180 MB a day. Its link uploads 25 MB a minute and is available for 2 hours a day. How many MB remain un-uploaded at the end of 5 days?", 0),
    ("A technician drives to site A, 42 km away, then 18 km further to site B, then 25 km further to site C, and drives straight home along the same road. Fuel costs 0.2 per km. What is the total fuel cost?", 34),
]
SUFFIX = "\nEnd your reply with a line of the form 'Answer: <number>'."

def extract_number(text: str):
    """Pull the final numeric answer out of a reply: the 'Answer:' line if present, else the last number."""
    m = re.search(r"Answer:\s*\$?\s*(-?[\d,]*\.?\d+)", text)
    found = m.group(1) if m else (re.findall(r"-?[\d,]*\.?\d+", text) or [None])[-1]
    if found is None:
        return None
    value = float(found.replace(",", ""))
    return int(value) if value == int(value) else value

print(len(PROBLEMS), "problems")
print(extract_number("Total is 2,100 - 280 = 1,820 kg.\nAnswer: 1,820"), extract_number("about 34.0"), extract_number("no digits here"))

```text
10 problems
1820 34 None
```

Three conditions: plain (low effort), self-consistency with five low-effort samples, and one high-effort answer. Current Claude models reject non-default temperatures, but the default sampling is already random, so repeated calls give different chains. Expect roughly 25 calls; keep your numbers for session 3.

In [ ]:
# Needs an API key (see the setup cell)
from collections import Counter

def plain(q):                                  # the fast path: low effort, answer only
    text, usage, secs = ask(q + SUFFIX, effort="low", max_tokens=2000)
    return extract_number(text), [usage], secs

def self_consistency(q, n=5):                  # sample n times at low effort, majority vote
    votes, usages, secs = [], [], 0.0
    for _ in range(n):
        text, usage, s = ask(q + SUFFIX, effort="low", max_tokens=2000)
        votes.append(extract_number(text)); usages.append(usage); secs += s
    return Counter(votes).most_common(1)[0][0], usages, secs

def with_thinking(q):                          # one sample, high effort
    text, usage, secs = ask(q + SUFFIX, effort="high", max_tokens=8000)
    return extract_number(text), [usage], secs

def evaluate(method, **kw):
    right, tokens, dollars, seconds = 0, 0, 0.0, 0.0
    for q, gold in PROBLEMS:
        pred, usages, secs = method(q, **kw)
        right += (pred == gold); seconds += secs
        tokens += sum(u.output_tokens for u in usages); dollars += sum(cost(u) for u in usages)
    label = f"{method.__name__} {kw if kw else ''}".strip()
    print(f"{label:>24}: {right}/{len(PROBLEMS)} correct, {tokens:>6} output tokens, "
          f"${dollars:.4f}, {seconds:.0f}s")

evaluate(plain)
evaluate(self_consistency, n=5)
evaluate(with_thinking)

Do not expect drama on these easy problems: the plain path may already get most of them, and the expensive conditions cost several times more for a point or two. Make the problems harder and the gap opens.

### Hands-on: map where voting helps

1. Using `vote_accuracy`, sweep `p` over 0.2 to 0.7 and the share of the wrong-answer mass on the single most common wrong answer over 0.3 to 0.9, splitting the rest equally over two other distractors. For each cell compute the accuracy with N = 15 minus `p`: the gain from voting.
2. Print the grid and plot it as a heatmap.
3. Find the boundary between gains and losses and check that it is the line `p = q`, where `q` is the top distractor's probability `(1 - p) * share`.
4. Optional, with an API key: rerun the real experiment with N = 1, 3, 5 and 9 and note where accuracy stops improving.

#### Solution: the gain from voting as a grid

In [ ]:
rng = np.random.default_rng(0)
ps = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
lumps = [0.3, 0.5, 0.7, 0.9]            # share of the wrong mass on the single most common wrong answer
gain = np.zeros((len(ps), len(lumps)))
for i, p in enumerate(ps):
    for j, lump in enumerate(lumps):
        wrong = [(1 - p) * lump, (1 - p) * (1 - lump) / 2, (1 - p) * (1 - lump) / 2]
        gain[i, j] = vote_accuracy(p, wrong, 15, trials=5000) - p

print("gain from voting with N=15 (accuracy minus p); rows p, columns lump share")
print("      " + "".join(f"{l:>8.1f}" for l in lumps))
for p, row in zip(ps, gain):
    print(f"p={p:.1f} " + "".join(f"{g:>+8.2f}" for g in row))

```text
gain from voting with N=15 (accuracy minus p); rows p, columns lump share
           0.3     0.5     0.7     0.9
p=0.2    -0.08   -0.10   -0.15   -0.19
p=0.3    +0.09   +0.05   -0.10   -0.22
p=0.4    +0.30   +0.24   +0.06   -0.11
p=0.5    +0.39   +0.35   +0.24   +0.07
p=0.6    +0.38   +0.36   +0.32   +0.23
p=0.7    +0.30   +0.29   +0.28   +0.26
```

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 3.5))
im = ax.imshow(gain, origin="lower", cmap="RdBu", vmin=-0.4, vmax=0.4, aspect="auto")
ax.set_xticks(range(len(lumps)), [f"{l:.1f}" for l in lumps]); ax.set_xlabel("share of wrong mass on top distractor")
ax.set_yticks(range(len(ps)), [f"{p:.1f}" for p in ps]); ax.set_ylabel("p (single-sample accuracy)")
for i in range(len(ps)):
    for j in range(len(lumps)):
        ax.text(j, i, f"{gain[i, j]:+.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="accuracy gain, N = 15"); ax.set_title("Where majority voting helps")
plt.tight_layout(); plt.show()

The top-left corner is negative because at p = 0.2 even an evenly spread error beats the correct answer (q = 0.24). Along the diagonal where p is close to q the gain is near zero and noisy: that is the regime where voting wastes money. The large positive region, a solver that is usually right and wrong in varied ways, is where self-consistency belongs.

---

## Session 3: Prompting a model that thinks

Give the goal, the constraints and what a good answer looks like, then let the model plan. Step-by-step instructions written for chat models make a reasoning model spend its thinking on obeying you.

*Micromanaged*

```text
Step 1: list every instrument and its last calibration date.
Step 2: for each instrument, add 90 days to get the due date.
Step 3: sort the due dates.
Step 4: assign the earliest three to next week, Monday to Wednesday.
Step 5: write the schedule as a table.
```

*Goal, constraints, success criteria*

```text
Produce next week's calibration schedule for the instruments listed below.

Constraints:
- An instrument is due 90 days after its last calibration; overdue ones come first.
- One technician, at most two calibrations a day, none on Friday.
- Sites A and C cannot be visited on the same day (4 hours apart).

A good schedule has no overdue instrument left unscheduled if capacity allows,
explains any that had to wait, and is a table with columns day, site, instrument.
If the data are inconsistent (a future calibration date), say so instead of guessing.

<instruments>
...
</instruments>
```

### Give it a way to check its work

A tool turns a belief into a fact. The model below gets a safe arithmetic tool and, with thinking on, thinks, calls it, reads the result and continues. Note that `resp.content` goes back unchanged so the thinking blocks travel with the tool calls.

In [ ]:
# Needs an API key (see the setup cell)
import ast, operator

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.USub: operator.neg}

def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression safely: numbers and + - * / only, no names, no calls."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    try:
        return str(ev(ast.parse(expression, mode="eval").body))
    except Exception as e:
        return f"error: {e}"

tools = [{"name": "calculate",
          "description": "Evaluate an arithmetic expression exactly. Use it to check every calculation before you answer.",
          "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}},
                           "required": ["expression"]}}]

messages = [{"role": "user", "content": PROBLEMS[9][0] + SUFFIX}]
while True:
    resp = client.messages.create(model=MODEL, max_tokens=8000, thinking={"type": "adaptive"},
                                  output_config={"effort": "medium"}, tools=tools, messages=messages)
    messages.append({"role": "assistant", "content": resp.content})   # thinking blocks go back unchanged
    if resp.stop_reason != "tool_use":
        break
    results = []
    for b in resp.content:
        if b.type == "tool_use":
            value = calculate(b.input["expression"])
            print("checked:", b.input["expression"], "=", value)
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": value})
    messages.append({"role": "user", "content": results})
print("".join(b.text for b in resp.content if b.type == "text"))
print("gold:", PROBLEMS[9][1])

### Structured output after thinking

The model thinks freely, then the final block is JSON matching the schema. The confidence field is imperfect but useful for routing.

In [ ]:
# Needs an API key (see the setup cell)
from pydantic import BaseModel, Field

class Verdict(BaseModel):
    answer: int
    confidence: float = Field(ge=0, le=1, description="Your honest probability that the answer is right")
    assumptions: list[str] = Field(description="Anything you had to assume that the question did not state")

resp = client.messages.parse(
    model=MODEL, max_tokens=8000,
    thinking={"type": "adaptive"}, output_config={"effort": "medium"},
    output_format=Verdict,                       # the final text block is JSON matching the schema
    messages=[{"role": "user", "content": PROBLEMS[4][0]}],
)
v = resp.parsed_output                           # a validated Verdict
print(v.answer, v.confidence, v.assumptions)
print("gold:", PROBLEMS[4][1], "| output tokens:", resp.usage.output_tokens)

### The visible reasoning is not the cause

The chain of thought is generated text; nothing forces the answer to be caused by it, and providers return summaries. Use it for debugging and intuition, never as the sole basis for a safety or compliance decision.

## Evaluating reasoning systems

### pass@k and pass^k

pass@k: at least one of k attempts is correct (needs a verifier to pick it). pass^k: all k attempts are correct (what a user who runs the same task every day experiences).

In [ ]:
from math import comb

def pass_at_k(n, c, k):
    """Unbiased estimate of pass@k from n samples of which c were correct (Chen et al. 2021)."""
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

rng = np.random.default_rng(1)
per_problem_p = rng.beta(2, 2, size=40)            # 40 problems of varying difficulty
n = 20
correct_counts = rng.binomial(n, per_problem_p)    # n samples per problem
p_hat = correct_counts / n
for k in [1, 2, 5, 10]:
    pk = np.mean([pass_at_k(n, c, k) for c in correct_counts])
    ppk = np.mean(p_hat ** k)                      # pass^k: all k attempts succeed
    print(f"k={k:>2}   pass@k {pk:.3f}   pass^k {ppk:.3f}")

```text
k= 1   pass@k 0.550   pass^k 0.550
k= 2   pass@k 0.754   pass^k 0.356
k= 5   pass@k 0.921   pass^k 0.147
k=10   pass@k 0.965   pass^k 0.066
```

In [ ]:
ks = range(1, 11)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ks, [np.mean([pass_at_k(n, c, k) for c in correct_counts]) for k in ks], marker="o", label="pass@k (at least one of k)")
ax.plot(ks, [np.mean(p_hat ** k) for k in ks], marker="s", label="pass^k (all k)")
ax.set_xlabel("k"); ax.set_ylabel("probability"); ax.set_title("Same system, two very different stories")
ax.legend(); plt.tight_layout(); plt.show()

### Accuracy per dollar and per second

Put every configuration on the same axes and keep the Pareto frontier. Here cost is in units of one plain sample, with the verifier charged 20% per sample; do the same with the real numbers from session 2.

In [ ]:
def pareto(points):
    """Keep the (name, cost, accuracy) points that no cheaper point matches or beats."""
    kept, best = [], -1.0
    for name, cost, acc in sorted(points, key=lambda t: t[1]):
        if acc > best:
            kept.append((name, cost, acc)); best = acc
    return kept

rng = np.random.default_rng(0)
points = []
for n in [1, 3, 5, 9, 15]:
    points.append((f"vote N={n}", n, vote_accuracy(0.6, spread, n)))
    points.append((f"best-of N={n}", n * 1.2, best_of_n_accuracy(0.6, n, 1.5)))   # the verifier adds 20% per sample
for name, c, a in pareto(points):
    print(f"{name:>13}: cost {c:5.1f}   accuracy {a:.3f}   accuracy per unit cost {a / c:.3f}")

```text
     vote N=1: cost   1.0   accuracy 0.597   accuracy per unit cost 0.597
  best-of N=1: cost   1.2   accuracy 0.604   accuracy per unit cost 0.503
     vote N=3: cost   3.0   accuracy 0.717   accuracy per unit cost 0.239
  best-of N=3: cost   3.6   accuracy 0.844   accuracy per unit cost 0.235
  best-of N=5: cost   6.0   accuracy 0.908   accuracy per unit cost 0.151
     vote N=9: cost   9.0   accuracy 0.941   accuracy per unit cost 0.105
  best-of N=9: cost  10.8   accuracy 0.952   accuracy per unit cost 0.088
    vote N=15: cost  15.0   accuracy 0.988   accuracy per unit cost 0.066
```

### Routing

Send each request to the cheapest model likely to handle it. A rule-based router is the simplest; better ones use a signal: a cheap difficulty rating, the fast model's confidence, or disagreement between two cheap samples.

In [ ]:
HARD_SIGNS = [
    r"(\d[\d.,]*\D+){3,}\d",                             # three or more numbers to combine
    r"\b(why|plan|schedule|optimi[sz]e|prove|debug|trade-?off)\b",
    r"\b(constraints?|subject to|at most|at least|must not)\b",
    r"```",                                              # a code block
]

def route(question: str) -> str:
    """Return 'fast' or 'reasoning'. It errs towards 'reasoning': a wrong 'fast'
    costs accuracy, a wrong 'reasoning' only costs money."""
    hard = any(re.search(p, question, flags=re.I | re.S) for p in HARD_SIGNS)
    return "reasoning" if hard or len(question.split()) > 60 else "fast"

questions = ["What is the capital of Peru?",
             "Translate 'good morning' into French.",
             PROBLEMS[9][0],
             "Plan a 3-day calibration schedule for 5 sites subject to the constraints below: ..."]
for q in questions:
    print(f"{route(q):>9}: {q[:72]}")

```text
     fast: What is the capital of Peru?
     fast: Translate 'good morning' into French.
reasoning: A technician drives to site A, 42 km away, then 18 km further to site B,
reasoning: Plan a 3-day calibration schedule for 5 sites subject to the constraints
```

### Failure modes

Overthinking easy questions, confident wrong chains, budget exhaustion (`stop_reason == "max_tokens"` with no text), reward hacking such as special-casing tests, accuracy falling with more samples, and unfaithful reasoning. The lesson page has the symptom, cause and fix for each.

### Hands-on: a disagreement router

1. Simulate 3,000 questions. Draw the fast model's per-question accuracy from a Beta(4, 2) distribution and give the reasoning model 0.3 more, capped at 0.98. The fast model costs 1 per call and the reasoning model 15. Wrong answers are spread uniformly over three distractors.
2. Implement three policies: always fast, always reasoning, and the disagreement router (two fast samples; escalate only if they differ).
3. Report accuracy, cost per question and the escalation rate. Where does the router sit between the two baselines?
4. Vary it: three fast samples with a majority vote, escalating only on a three-way disagreement; a cheaper reasoning model; a fast model that is more often wrong in the same way (fewer distractors). Which changes help the router most?
5. Optional, with an API key: implement the router with `MODEL_FAST` and `MODEL` on the session 2 problems and add its row to the session 2 table.

#### Solution: simulate the router, then run it for real

In [ ]:
rng = np.random.default_rng(3)
M = 3000
p_fast = rng.beta(4, 2, size=M)                     # per-question accuracy of the fast model
p_reason = np.clip(p_fast + 0.3, 0, 0.98)           # the reasoning model is better, most where fast is weak
C_FAST, C_REASON, K = 1.0, 15.0, 3                  # relative costs; wrong answers spread over K distractors

def sample(p):
    """A sample from a model with accuracy p: 0 is the right answer, 1..K are wrong ones."""
    return 0 if rng.random() < p else rng.integers(1, K + 1)

def run(policy):
    right, cost, escalated = 0, 0.0, 0
    for pf, pr in zip(p_fast, p_reason):
        if policy == "fast":
            right += sample(pf) == 0; cost += C_FAST
        elif policy == "reasoning":
            right += sample(pr) == 0; cost += C_REASON
        else:                                          # two fast samples; escalate only if they disagree
            a, b = sample(pf), sample(pf); cost += 2 * C_FAST
            if a == b:
                right += a == 0
            else:
                right += sample(pr) == 0; cost += C_REASON; escalated += 1
    print(f"{policy:>12}: accuracy {right / M:.3f}   cost per question {cost / M:5.2f}   escalated {escalated / M:.0%}")

for policy in ["fast", "reasoning", "disagreement"]:
    run(policy)

```text
        fast: accuracy 0.671   cost per question  1.00   escalated 0%
   reasoning: accuracy 0.898   cost per question 15.00   escalated 0%
disagreement: accuracy 0.888   cost per question  9.27   escalated 48%
```

The router recovers nearly all of the reasoning model's accuracy at well under two thirds of its cost, escalating about half the questions. Three samples with a vote escalate less and cost less at a small loss of accuracy; fewer distractors hurt the router because wrong answers then agree with each other more often (the lumped regime from session 2). The real version is a few lines on top of `ask()`:

In [ ]:
# Needs an API key (see the setup cell)
def disagreement_router(q):
    """Two cheap samples; return their answer if they agree, else escalate to the reasoning model."""
    a_text, a_use, s1 = ask(q + SUFFIX, effort="low", max_tokens=2000, model=MODEL_FAST)
    b_text, b_use, s2 = ask(q + SUFFIX, effort="low", max_tokens=2000, model=MODEL_FAST)
    a, b = extract_number(a_text), extract_number(b_text)
    if a == b:
        return a, [a_use, b_use], s1 + s2
    text, use, s3 = ask(q + SUFFIX, effort="high", max_tokens=8000)
    return extract_number(text), [a_use, b_use, use], s1 + s2 + s3

evaluate(disagreement_router)      # compare with the three rows from session 2

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/12-reasoning-models.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.